<a href="https://colab.research.google.com/github/CallMeGuptaji/AnyDLBot/blob/master/mycolab_desktop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖥️ Cloud Desktop — Google Colab

[![GitHub](https://img.shields.io/badge/GitHub-M5Develop%2FCloud--Desktop-181717?logo=github)](https://github.com/M5Develop/Cloud-Desktop-Colab-kaggle)
[![License: GPL v3](https://img.shields.io/badge/License-GPLv3-blue.svg)](https://github.com/M5Develop/Cloud-Desktop-Colab-kaggle/blob/main/LICENSE)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/M5Develop/Cloud-Desktop-Colab-kaggle/blob/main/kaggle_desktop.ipynb)

Run a full XFCE4 desktop on Google Colab and access it from any browser — no VPN, no credit card, no local storage needed.

**Stack:** `Xvfb` + `XFCE4` + `x11vnc` + `noVNC` + `localhost.run` + `Brave Browser`

**Notes:**
- ✅ Colab (Xubuntu) already has XFCE4, x11vnc, and websockify pre-installed
- ✅ Brave installs without `snap`
- ⏱️ Free tier session lasts ~12 hours

**Use cases:**
- 📤 Upload large model files (7GB+) to CivitAI using Colab's bandwidth — not yours
- 🖥️ Run any GUI application on a free cloud environment
- 🌐 Browse the web through a remote Linux desktop

> ⚠️ Run cells **in order**, one by one.

## 📦 Cell 1 — Install Dependencies
> Colab already has most packages pre-installed — this just ensures everything is ready.

In [1]:
!apt-get update -qq
!apt-get install -y -qq xfce4 x11vnc xvfb dbus-x11 novnc websockify openssh-client 2>/dev/null
print('✅ All dependencies ready!')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 122813 files and directories currently installed.)
Removing fuse (2.9.9-8.1build1) ...
Selecting previously unselected package liblocale-gettext-perl.
(Reading database ... 122803 files and directories currently installed.)
Preparing to unpack .../liblocale-gettext-perl_1.07-6ubuntu5_amd64.deb ...
Unpacking liblocale-gettext-perl (1.07-6ubuntu5) ...
Selecting previously unselected package keyboard-configuration.
Preparing to unpack .../keyboard-configuration_1.226ubuntu1.1_all.deb ...
Unpacking keyboard-configuration (1.226ubuntu1.1) ...
Selecting previously unselected package libfuse3-3:amd64.
Preparing to unpack .../libfuse3-3_3.14.0-5build1_amd64.deb ...
Unpacking libfuse3-3:amd64 (3.14.0-5build1) ...
Selecting previously unselected package fuse3.


## 🌐 Cell 2 — Install Brave Browser
> Installs via official `.deb` — no `snap` required.

In [2]:
import subprocess
subprocess.run('curl -fsS https://dl.brave.com/install.sh | sh', shell=True)
print('✅ Brave installed!')

✅ Brave installed!


## 🖥️ Cell 3 — Start Virtual Display + XFCE4

In [3]:

import os, time, subprocess, socket

os.environ["DISPLAY"] = ":99"

subprocess.Popen(
    ["Xvfb", ":99", "-screen", "0", "1280x800x24", "-ac"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

def check_port(port):
    s = socket.socket()
    s.settimeout(2)
    try:
        s.connect(("127.0.0.1", port))
        return True
    except OSError:
        return False
    finally:
        s.close()

if not check_port(6000 + 99):
    print("Checking Xvfb process...")
    subprocess.run("pgrep -af Xvfb", shell=True)

subprocess.Popen(
    "dbus-launch --exit-with-session startxfce4",
    shell=True,
    env=os.environ.copy(),
    stdout=open("/tmp/xfce.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)
print("XFCE startup attempted.")
print("Display:", os.environ["DISPLAY"])


✅ Xvfb running!
✅ XFCE4 running!


## 📡 Cell 4 — Start VNC Server

In [4]:

import subprocess, time, socket, secrets

# VNC passwords traditionally use only the first 8 characters.
VNC_PASSWORD = secrets.token_hex(4)

subprocess.run(
    ["x11vnc", "-storepasswd", VNC_PASSWORD, "/root/.vncpasswd"],
    check=True,
    stdout=subprocess.DEVNULL
)

vnc = subprocess.Popen(
    ["x11vnc", "-display", ":99",
     "-rfbauth", "/root/.vncpasswd",
     "-rfbport", "5900", "-forever", "-shared", "-noxdamage"],
    stdout=open("/tmp/x11vnc.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

s = socket.socket()
s.settimeout(2)
try:
    s.connect(("127.0.0.1", 5900))
    print("VNC server: OPEN")
    print("VNC password:", VNC_PASSWORD)
except OSError:
    print("VNC server FAILED. Check /tmp/x11vnc.log")
finally:
    s.close()


✅ VNC server running (password: 12345678)


## 🖱️ Cell 5 — Start noVNC

In [5]:

import subprocess, time, socket, os

websockify = subprocess.Popen(
    ["websockify", "--web=/usr/share/novnc/",
     "6080", "localhost:5900"],
    stdout=open("/tmp/websockify.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

s = socket.socket()
s.settimeout(2)
try:
    s.connect(("127.0.0.1", 6080))
    print("noVNC web server: OPEN")
except OSError:
    print("noVNC FAILED. Check /tmp/websockify.log")
finally:
    s.close()


✅ noVNC running on port 6080!


## 🔗 Cell 6 — Create Public Tunnel via localhost.run
> ✅ Free | ✅ No account | ✅ No credit card | Uses SSH only

In [6]:

import subprocess, time, os

log_path = "/tmp/localhost-run.log"

tunnel = subprocess.Popen(
    ["ssh",
     "-o", "StrictHostKeyChecking=no",
     "-o", "ExitOnForwardFailure=yes",
     "-o", "ServerAliveInterval=30",
     "-R", "80:localhost:6080",
     "nokey@localhost.run"],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT
)

print("Waiting for tunnel...")
for _ in range(30):
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path) as f:
            log = f.read()
        if "https://" in log and ".lhr.life" in log:
            print(log)
            break
else:
    print("Tunnel not confirmed. Log:")
    with open(log_path) as f:
        print(f.read())


⏳ Starting tunnel...
⚠️ URL not ready — run Cell 7 to check the log


## 🔍 Cell 7 — Show Full Tunnel Log (if URL didn't appear)

In [7]:
with open('/root/tunnel.log', 'r') as f:
    print(f.read())

Pseudo-terminal will not be allocated because stdin is not a terminal.

Welcome to localhost.run!

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/



** your connection id is 136.112.191.230:58362, please mention it if you send me a message about an issue. **

ff9279a1c0e8dc.lhr.life tunneled with tls termination, https://ff9279a1c0e8dc.lhr.life
create an account and add your key for a longer lasting domain name. see https://localhost.run/docs/forever-free/ for more information.
Open your tunnel address on your mobile with this QR:

                                       

In [10]:

import socket
import subprocess
import os

print("=== VNC / noVNC processes ===")
subprocess.run(
    "ps aux | grep -E 'websockify|novnc|Xvnc|Xtigervnc|x11vnc' | grep -v grep",
    shell=True
)

print("\n=== Listening ports ===")
subprocess.run(
    "ss -ltnp | grep -E ':5900|:5901|:6080' || true",
    shell=True
)

print("\n=== Tunnel log ===")
if os.path.exists("/root/tunnel.log"):
    with open("/root/tunnel.log") as f:
        print(f.read()[-5000:])
else:
    print("Tunnel log does not exist; tunnel may not have started in this runtime.")

print("\n=== Port checks ===")
for port in (5900, 5901, 6080):
    s = socket.socket()
    s.settimeout(2)
    try:
        s.connect(("127.0.0.1", port))
        print(f"Port {port}: OPEN")
    except OSError:
        print(f"Port {port}: CLOSED")
    finally:
        s.close()


=== VNC / noVNC processes ===

=== Listening ports ===

=== Tunnel log ===
                                   
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
                                                      
         

## 🚀 Cell 8 — Launch Brave Browser on the Desktop

In [8]:
import subprocess, os
subprocess.Popen('DISPLAY=:99 brave-browser --no-sandbox', shell=True,
    env={**os.environ, 'DISPLAY': ':99'},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✅ Brave launched on the desktop!')

✅ Brave launched on the desktop!


## ⬇️ Cell 9 (Optional) — Download Model from HuggingFace
> Server-to-server download — **zero local bandwidth used!** 😎

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

REPO_ID  = 'your-username/your-model'   # ← Change this
FILENAME = 'your-model.safetensors'     # ← Change this

print(f'⏳ Downloading {FILENAME} from HuggingFace...')
print('(Server-to-server — no local bandwidth used 😎)')

path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir='/root/')

os.makedirs('/root/Desktop', exist_ok=True)
shutil.copy(path, f'/root/Desktop/{FILENAME}')

print(f'✅ Downloaded: {path}')
print('✅ Copied to Desktop — ready to upload!')

## 🛑 Cell 10 — Stop Everything

In [ ]:
import subprocess
subprocess.run('pkill x11vnc; pkill Xvfb; pkill xfce4-session; pkill websockify; pkill ssh', shell=True)
print('🛑 All processes stopped!')